# 01 — Analyse Exploratoire des Données

Ce notebook explore les deux datasets avant tout entraînement.
L'objectif est de comprendre la structure des données, détecter des biais potentiels
et formuler des hypothèses sur les difficultés de classification.

| Dataset | Langue | Classes | Volume | Source |
|---------|--------|---------|--------|--------|
| **20 Newsgroups** | Anglais | 8 (sur 20) | ~4 700 train / ~3 100 test | Public — sklearn |
| **Mails clubs sportifs** | Français | 8 | ~1 440 train / ~360 test | Synthétique LLM |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
import re

from src.data.newsgroups import load_newsgroups, SELECTED_CLASSES
from src.data.email_dataset import load_emails, CLASSES

# Palette de couleurs cohérente sur tout le notebook
PALETTE = px.colors.qualitative.Set2
TEMPLATE = 'plotly_white'

---
## 1. Dataset 1 — 20 Newsgroups (validation scientifique)

Dataset public de référence. On retient 8 classes couvrant sciences, informatique,
sport et politique pour garantir la reproductibilité du benchmark.

In [2]:
train_ng, test_ng = load_newsgroups()
all_ng = pd.concat([train_ng.assign(split='train'), test_ng.assign(split='test')])

print(f"Train : {len(train_ng):,} documents | Test : {len(test_ng):,} documents")
print(f"Classes : {train_ng['label'].nunique()}")
print(f"\nDistribution train :")
print(train_ng['label'].value_counts().to_string())

Train : 4,310 documents | Test : 2,868 documents
Classes : 8

Distribution train :
label
rec.sport.hockey           580
sci.med                    576
sci.space                  576
comp.graphics              567
comp.os.ms-windows.misc    564
rec.autos                  556
talk.politics.guns         532
talk.religion.misc         359


In [3]:
# Distribution des classes — tri par fréquence
counts = train_ng['label'].value_counts().reset_index()
counts.columns = ['classe', 'count']

fig = px.bar(
    counts, x='count', y='classe', orientation='h',
    color='count', color_continuous_scale='Blues',
    title='20 Newsgroups — Distribution des classes (train)',
    labels={'count': 'Nombre de documents', 'classe': ''},
    template=TEMPLATE,
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=350, margin=dict(l=0))
fig.show()

In [4]:
# Longueur des documents — violin plot par classe
train_ng['n_tokens_approx'] = train_ng['text'].str.split().str.len()

fig = px.violin(
    train_ng, x='label', y='n_tokens_approx',
    box=True, points=False,
    color='label', color_discrete_sequence=PALETTE,
    title='Longueur des documents par classe (tokens approximatifs)',
    labels={'n_tokens_approx': 'Nombre de mots', 'label': ''},
    template=TEMPLATE,
)
fig.update_layout(showlegend=False, height=400)
fig.add_hline(y=512, line_dash='dash', line_color='red',
              annotation_text='Limite CamemBERT (512 tokens)',
              annotation_position='top right')
fig.show()

In [5]:
# Proportion de documents dépassant 512 tokens
over_512 = (train_ng['n_tokens_approx'] > 512).mean()
print(f"Documents dépassant 512 tokens : {over_512:.1%}")
print(f"→ Ces documents sont tronqués pour CamemBERT, ce qui peut affecter")
print(f"  la classification si les informations clés sont en fin de texte.")

# Aperçu de quelques documents
print("\n" + "="*60)
for label in SELECTED_CLASSES[:2]:
    sample = train_ng[train_ng['label'] == label]['text'].iloc[0]
    print(f"\n[{label}]\n{sample[:250]}...")

Documents dépassant 512 tokens : 6.6%
→ Ces documents sont tronqués pour CamemBERT, ce qui peut affecter
  la classification si les informations clés sont en fin de texte.


[sci.med]
(Disclaimer: I'm a sufferer, not a doctor.)

I'm not sure there's a really sharp distinction between allergic and
vasomotor rhinitis.  Basically, vasomotor rhinitis means your nose is
stuffy when it has no reason to be (not even an identifiable aller...

[sci.space]

Sure.  Contact the World Space Foundation.  They're listed in the sci.space
Frequently Asked Questions file, which I'll excerpt.

    WORLD SPACE FOUNDATION - has been designing and building a solar-sail
    spacecraft for longer than any similar gr...


---
## 2. Dataset 2 — Mails synthétiques (cas d'usage métier)

Dataset original de ~1 800 mails de clubs sportifs français, généré via LLM
avec variations contrôlées de longueur, ton, urgence, persona et sport.

In [6]:
train_em, test_em = load_emails()
full_em = pd.read_csv('../data/processed/emails.csv')

print(f"Total : {len(full_em):,} mails | Train : {len(train_em):,} | Test : {len(test_em):,}")
print(f"Doublons : {full_em['text'].duplicated().sum()} | Vides : {full_em['text'].isna().sum()}")

Total : 1,800 mails | Train : 1,440 | Test : 360
Doublons : 0 | Vides : 0


In [7]:
# Distribution des catégories
counts_em = full_em['label'].value_counts().reset_index()
counts_em.columns = ['categorie', 'count']

fig = px.bar(
    counts_em, x='count', y='categorie', orientation='h',
    color='count', color_continuous_scale='Greens',
    title='Mails synthétiques — Distribution des catégories',
    labels={'count': 'Nombre de mails', 'categorie': ''},
    template=TEMPLATE,
)
fig.update_coloraxes(showscale=False)
fig.update_layout(height=350)
fig.show()

In [8]:
# Longueur des mails par catégorie
full_em['n_words'] = full_em['text'].str.split().str.len()

fig = px.box(
    full_em, x='label', y='n_words',
    color='label', color_discrete_sequence=PALETTE,
    title='Longueur des mails par catégorie (en mots)',
    labels={'n_words': 'Nombre de mots', 'label': ''},
    template=TEMPLATE,
)
fig.update_layout(showlegend=False, height=400)
fig.show()

In [9]:
# Variations contrôlées : sport, ton, persona
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Par sport', 'Par ton', 'Par persona expéditeur']
)

for col_idx, col in enumerate(['sport', 'tone', 'persona'], 1):
    if col not in full_em.columns:
        continue
    counts = full_em[col].value_counts()
    fig.add_trace(
        go.Bar(x=counts.values, y=counts.index, orientation='h',
               marker_color=PALETTE[col_idx - 1], showlegend=False),
        row=1, col=col_idx
    )

fig.update_layout(
    title='Variations contrôlées dans le dataset synthétique',
    height=350, template=TEMPLATE,
)
fig.show()

In [10]:
# Exemples de mails par catégorie
print("EXEMPLES DE MAILS PAR CATÉGORIE\n" + "="*60)
for cat in CLASSES[:4]:
    sample = full_em[full_em['label'] == cat]['text'].iloc[0]
    print(f"\n[{cat.upper()}]\n{sample[:300]}...\n")

EXEMPLES DE MAILS PAR CATÉGORIE

[INSCRIPTION]
Bonjour,

Je dois inscrire rapidement un joueur pour la saison, c'est urgent. Pouvez-vous me dire combien ça coûte et quels documents il faut prévoir ? Il peut commencer dès cette semaine.

Merci de me répondre au plus vite.

Cordialement,
Marc...


[SPONSOR]
Madame, Monsieur,

J'espère que vous allez bien. Je m'adresse à vous en tant qu'entraîneur bénévole au sein du club d'athlétisme de Villefranche-sur-Saône depuis maintenant 8 ans. Notre association a beaucoup grandi ces dernières années et nous accueillons actuellement une soixantaine d'adhérents, d...


[ARBITRAGE-OFFICIELS]
Bonjour,

Je vous contacte en urgence concernant le match de ce soir. L'arbitre désigné ne pourra pas venir, il m'a prévenu y a une heure. Faut vraiment trouver quelqu'un d'autre rapidement sinon on pourra pas jouer.

Est-ce que vous avez un arbitre disponible en dernière minute ? On compte sur vous...


[PARENT]
Bonjour,

Je vous écris car mon fils Thomas ne pou

---
## 3. Comparaison des deux datasets

Les deux datasets ont des profils très différents qui vont influencer les résultats.

In [11]:
# Comparaison des longueurs
train_ng['dataset'] = '20 Newsgroups'
train_em['n_tokens_approx'] = train_em['text'].str.split().str.len()
train_em['dataset'] = 'Emails sportifs'

compare = pd.concat([
    train_ng[['n_tokens_approx', 'dataset']],
    train_em[['n_tokens_approx', 'dataset']]
])

fig = px.histogram(
    compare, x='n_tokens_approx', color='dataset',
    barmode='overlay', opacity=0.7,
    nbins=50, log_y=True,
    title='Comparaison des longueurs de texte — Newsgroups vs Emails',
    labels={'n_tokens_approx': 'Longueur (mots)', 'count': 'Fréquence (log)'},
    color_discrete_sequence=['#636EFA', '#00CC96'],
    template=TEMPLATE,
)
fig.add_vline(x=128, line_dash='dot', line_color='orange',
              annotation_text='max_seq_length SetFit (128)')
fig.add_vline(x=512, line_dash='dash', line_color='red',
              annotation_text='max CamemBERT (512)')
fig.update_layout(height=400)
fig.show()

In [12]:
# Tableau récapitulatif
summary = pd.DataFrame({
    'Dataset': ['20 Newsgroups', 'Emails sportifs'],
    'Langue': ['Anglais', 'Français'],
    'Classes': [8, 8],
    'Train': [len(train_ng), len(train_em)],
    'Test': [len(test_ng), len(test_em)],
    'Longueur médiane (mots)': [
        int(train_ng['n_tokens_approx'].median()),
        int(train_em['n_tokens_approx'].median())
    ],
    'Docs > 512 tokens': [
        f"{(train_ng['n_tokens_approx'] > 512).mean():.0%}",
        f"{(train_em['n_tokens_approx'] > 512).mean():.0%}",
    ],
    'Équilibre classes': ['Bon', 'Parfait (225/cat)'],
})
summary

,Dataset,Langue,Classes,Train,Test,Longueur médiane (mots),Docs > 512 tokens,Équilibre classes
0,20 Newsgroups,Anglais,8,4310,2868,83,7%,Bon
1,Emails sportifs,Français,8,1440,360,99,0%,Parfait (225/cat)
